# Pure Weber Cross Molecule: Symmetric Two-Line Initial Condition

This notebook demonstrates a four-body configuration in **pure Weber electrodynamics
(no Zöllner extension)** using an initial condition that is fully **symmetric
about two orthogonal lines**: the positive pair lies on the $x$-axis and the
negative pair lies on a perpendicular vertical line $x = R/2$.

Despite the simple geometric starting point, the system exhibits persistent
sub-critical oscillation of the positive pair — stabilised by the negative pair
and the bounce regularisation — for the full duration of the simulation.

## Physics Background

### Weber's Force Law and the Critical Radius

Weber's force between charges $q_i$, $q_j$ separated by $r$ is

$$F = \frac{q_i q_j}{r^2}\left(1 - \frac{\dot{r}^2}{2c^2} + \frac{r\ddot{r}}{c^2}\right)$$

The velocity-dependent terms introduce an **effective inertial mass**

$$\mu_{\text{eff}}(r) = \mu\left(1 - \frac{\rho}{r}\right), \qquad \rho = \frac{q_i q_j}{\mu c^2}$$

For two like charges ($q_i q_j > 0$), $\rho > 0$ is the **critical radius**.
At $r < \rho$ (sub-critical) the effective mass is **negative**: the pair attracts
instead of repelling.

### The Frauenfelder Non-Regularisability Result

Frauenfelder & Weber (2024) prove that a sub-critical like-charge pair with nonzero angular
momentum ($\ell \neq 0$) spirals inward with **infinitely many windings** in finite time — a
topological obstruction that **no regularisation can remove** (Theorem 2.1 of
[arXiv:2310.11560](https://link.springer.com/article/10.1007/s13324-024-00889-5)).

Only the head-on ($\ell = 0$) case admits regularisation, via a hard bounce reflection
(`regularization_collision_bounce_radius`) that continues the trajectory through $r = 0$.

### Symmetric Cross-Molecule Configuration

The initial condition places the two charge pairs on **two perpendicular lines**:

```
y
↑
|   N1(R/2, +rB/2)
|     |
P1 ──┼── P2    (x-axis, y = 0)
|     |
|   N2(R/2, -rB/2)
+──────────────→ x
```

- **Pair A** (positive, sub-critical): P1 at $(-r_A/2,\,0)$, P2 at $(+r_A/2,\,0)$ — on the $x$-axis.
- **Pair B** (negative, super-critical): N1 at $(R/2,\,+r_B/2)$, N2 at $(R/2,\,-r_B/2)$ — on the vertical line $x = R/2$.

Both pairs receive the same magnitude velocity in the $y$-direction:
pair A moves **up** (+y), pair B moves **down** (−y). This causes the pair
centres of mass to orbit each other while the internal relative velocity of each
pair is **initially zero** — placing the positive pair in the head-on ($\ell = 0$)
regime that admits bounce regularisation.

| Particle | Charge | Pair | Regime |
|---|---|---|---|
| P1 | $+q$ | A | Sub-critical ($r_A < \rho$) |
| P2 | $+q$ | A | Sub-critical ($r_A < \rho$) |
| N1 | $-q$ | B | Super-critical ($r_B > \rho$) |
| N2 | $-q$ | B | Super-critical ($r_B > \rho$) |

**No Zöllner extension** is used — all pair couplings $\kappa_{ij} = 1$.

**References**: Frauenfelder & Weber, *Anal. Math. Phys.* **14**:31 (2024); Weber,
Sixth Memoir (1871) §§9.8–9.17; see `research/theory/` and
`research/sub_critical_weber_research/`.

In [ ]:
using WeberElectrodynamics
using LinearAlgebra
using Printf
using Plots

## 1. Physical Parameters

In [ ]:
m  = 1.0    # particle mass
q  = 1.0    # charge magnitude
c  = 4.0    # Weber speed constant

μ  = m * m / (m + m)      # reduced mass = 0.5
ρ  = q^2  / (μ * c^2)    # critical radius = 0.125

# Pair A: positive, sub-critical (placed on x-axis)
r_A      = 0.050           # initial PP separation  (2r_A = 0.10 < ρ)
bounce_r = 0.015           # ℓ=0 bounce radius

# Pair B: negative, super-critical (placed on vertical line x = R/2)
r_B = 0.180                # initial NN separation  (2r_B = 0.36 > ρ)

# Cross-molecule geometry
R  = 0.250                 # horizontal offset: N pair at x = R/2 = 0.125
η  = 1.1                   # orbital speed factor (1 = Coulomb circular estimate)

# Orbital speed estimate: v_ref ~ sqrt(2q²/(mR)) from inter-pair Coulomb balance
v_ref = sqrt(2 * q^2 / (m * R))
v_orb = η * v_ref          # speed of each individual particle (y-direction)

# Integration
dt     = 2e-5
tspan  = (0.0, 2.0)
stride = 100

system = WeberSystem(4, 2)

@printf("Critical radius ρ = %.4f\n\n", ρ)
@printf("Pair A (positive, x-axis):   r_A = %.3f  < ρ  →  sub-critical   bounce_r = %.3f\n", r_A, bounce_r)
@printf("Pair B (negative, x=R/2):    r_B = %.3f  > ρ  →  super-critical\n\n", r_B)
@printf("Cross-molecule geometry:     R = %.3f   η = %.1f\n", R, η)
@printf("                             v_ref = %.4f   v_orb = %.4f\n\n", v_ref, v_orb)
@printf("Integration:                 dt = %.0e   tspan = (%.1f, %.1f)\n",
    dt, tspan[1], tspan[2])

## 2. Initial Conditions and Problem Setup

Pair A (positive) lies on the horizontal $x$-axis; pair B (negative) lies on the
vertical line $x = R/2$. Both pairs start with equal and opposite $y$-velocities so
their centres of mass orbit each other. Within each pair the initial
**relative velocity is zero** — the positive pair starts in purely radial ($\ell = 0$)
motion, satisfying the bounce regularisation condition.

In [ ]:
# Pair A on x-axis: P1 at (-r_A/2, 0), P2 at (+r_A/2, 0)
x1 = -r_A/2;  y1 = 0.0
x2 = +r_A/2;  y2 = 0.0

# Pair B on vertical line x = R/2: N1 at (R/2, +r_B/2), N2 at (R/2, -r_B/2)
x3 = R/2;     y3 = +r_B/2
x4 = R/2;     y4 = -r_B/2

# Pair A moves +y (up), pair B moves -y (down) → they orbit each other
q0 = [x1, y1, x2, y2, x3, y3, x4, y4]
p0 = [0.0, m*v_orb, 0.0, m*v_orb, 0.0, -m*v_orb, 0.0, -m*v_orb]

prob = WeberProblem(system, tspan, q0, p0;
    masses  = [m, m, m, m],
    charges = [q, q, -q, -q],
    c       = c,
    dt      = dt,
    regularization_enabled                 = false,
    regularization_collision_bounce_radius = bounce_r,
    zollner_enabled = false)

@printf("Kappas (all 1.0 — no Zöllner): %s\n\n",
    join([@sprintf("%.1f", k) for k in prob.kappas], ", "))

@printf("Initial positions (orthogonal lines):\n")
@printf("  P1 = (%.4f, %.4f)   P2 = (%.4f, %.4f)  [x-axis, y = 0]\n", x1, y1, x2, y2)
@printf("  N1 = (%.4f, %.4f)   N2 = (%.4f, %.4f)  [vertical, x = %.4f]\n", x3, y3, x4, y4, R/2)

@printf("\nInitial separations:\n")
@printf("  r_PP (pair A) = %.4f   (< ρ=%.4f → sub-critical, ratio %.2f)\n",
    r_A, ρ, r_A/ρ)
@printf("  r_NN (pair B) = %.4f   (> ρ=%.4f → super-critical, ratio %.2f)\n",
    r_B, ρ, r_B/ρ)

r_PN_near = sqrt((x2-x3)^2+(y2-y3)^2)   # nearest PN pair (P2 to N1)
r_PN_far  = sqrt((x1-x3)^2+(y1-y3)^2)   # farthest PN pair (P1 to N1)
@printf("  r_PN (near)   = %.4f   r_PN (far) = %.4f\n", r_PN_near, r_PN_far)

println("\nSolving...")
sol = solve(prob)
@printf("Done: %d steps, retcode = %s, t_end = %.4f\n",
    length(sol.t), string(sol.retcode), sol.t[end])

## 3. Trajectory

In [ ]:
traj = compute_trajectory_data(sol, 4, 2; stride=stride)

colors = [:firebrick, :darkorange, :royalblue, :cyan4]
labels = ["P1 (+q)" "P2 (+q)" "N1 (−q)" "N2 (−q)"]

pl = plot(
    title  = "Four-body trajectory (pure Weber, cross-molecule IC)",
    xlabel = "x", ylabel = "y",
    aspect_ratio = :equal,
    legend = :outertopright,
    size   = (800, 600))

for i in 1:4
    xs = traj.trajectories[i][:, 1]
    ys = traj.trajectories[i][:, 2]
    plot!(pl, xs, ys; label=labels[i], color=colors[i], lw=0.8, alpha=0.7)
    # Mark initial position with a filled circle
    scatter!(pl, [xs[1]], [ys[1]]; color=colors[i], ms=6, label=false)
end

# Annotate the two orthogonal initial lines
hline!(pl, [0.0]; ls=:dash, color=:gray, alpha=0.4, label="x-axis (pair A IC)")
vline!(pl, [R/2]; ls=:dot, color=:purple, alpha=0.4, label="x=R/2 (pair B IC)")

pl

## 4. Positive-Pair Separation and Energy Conservation

Two diagnostics confirm the balancing mechanism:

1. **$r_{12}(t)$**: the positive-pair separation oscillates with $r_{12} > 0$ throughout —
   the bounce prevents the sub-critical singularity.
2. **Energy conservation**: total Weber energy stays close to its initial value.

In [ ]:
en = compute_energy_timeseries(sol; stride=stride)
t_plot = sol.t[1:stride:end]

# r12 from strided trajectory
r12 = [sqrt((traj.trajectories[1][k,1] - traj.trajectories[2][k,1])^2 +
            (traj.trajectories[1][k,2] - traj.trajectories[2][k,2])^2)
       for k in axes(traj.trajectories[1], 1)]

# Energy percent deviation from initial value
E0    = en.total_energy[1]
E_pct = 100 .* (en.total_energy .- E0) ./ abs(E0)

p1 = plot(t_plot, r12;
    xlabel = "t", ylabel = "r₁₂",
    title  = "Positive-pair separation r₁₂(t)",
    color  = :firebrick, lw = 0.8, label = "r₁₂(t)")
hline!(p1, [ρ]; ls=:dash, color=:black, label="ρ (critical radius)")
hline!(p1, [bounce_r]; ls=:dot, color=:gray, label="bounce_r")

p2 = plot(t_plot, E_pct;
    xlabel = "t", ylabel = "ΔE / |E₀|  (%)",
    title  = "Energy conservation",
    color  = :steelblue, lw = 0.8, label = false)
hline!(p2, [0.0]; ls=:dash, color=:gray, label=false)

n_bounces = sum(r12[k] > r12[k-1] && r12[k-1] < 1.5*bounce_r
               for k in 2:length(r12))

@printf("r₁₂ range:      %.4f – %.4f   (bounce_r = %.4f, ρ = %.4f)\n",
    minimum(r12), maximum(r12), bounce_r, ρ)
@printf("Bounce count:   %d  (in strided data; true count higher)\n", n_bounces)
@printf("Energy drift:   %.3f%%  (max global error over full run)\n",
    en.statistics.global_error_percent_max)

plot(p1, p2; layout=(2,1), size=(800,600))